In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワーク生成 (Numba JIT)
# ==========================================
@njit
def build_ba_edges_fast(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    idx = 0; target_idx = 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    chosen = np.empty(m, dtype=np.int32)
    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node: already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx]

def build_clustered_network(N, m, weak_ties_ratio=0.002):
    N_A = N // 2
    N_B = N - N_A
    row_A, col_A = build_ba_edges_fast(N_A, m)
    row_B, col_B = build_ba_edges_fast(N_B, m)
    row_B = row_B + N_A; col_B = col_B + N_A

    row = np.concatenate([row_A, row_B])
    col = np.concatenate([col_A, col_B])

    num_weak_ties = max(1, int(N * weak_ties_ratio))
    wt_row = np.random.randint(0, N_A, num_weak_ties)
    wt_col = np.random.randint(N_A, N, num_weak_ties)

    row = np.concatenate([row, wt_row, wt_col])
    col = np.concatenate([col, wt_col, wt_row])

    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))

    deg_A = np.bincount(row_A, minlength=N_A)
    hub_A = np.argmax(deg_A)
    boundary_nodes = np.unique(np.concatenate([wt_row, wt_col]))

    return A_sparse, N_A, hub_A, boundary_nodes

# ==========================================
# 2. HJBコスト評価シミュレーション
# ==========================================
def run_hjb_simulation(N, A_sparse, N_A, hub_A, boundary_nodes, params, T_prev):
    dt = params['dt']
    steps = params['steps']
    T_expose = params['T_expose']

    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0

    # 初期値依存を排除した完全な乱数初期化
    np.random.seed()
    X = np.random.uniform(0, 0.2, N).astype(np.float32)
    D = np.random.uniform(0, 0.1, N).astype(np.float32)

    v_target = np.full(N, 0.3, dtype=np.float32)
    v_target[:N_A] = 1.0

    triad_boost = np.ones(N, dtype=np.float32)
    b = np.zeros(N, dtype=np.float32)

    history_D_B = []

    # コスト計算用変数
    social_loss = 0.0
    intervention_cost = 0.0

    for step in range(steps):
        # フェイクの発生
        if step < T_expose: b[hub_A] = 0.95
        else: b[hub_A] = 0.0

        # 介入タイミングでのトライアド（2-単体）注入
        u_t = 0.0
        if step >= T_prev and T_prev < steps:
            triad_boost[boundary_nodes] = params['triad_multiplier']
            u_t = params['u_cost_weight'] # 介入の計算コスト（仕事量）

        # SDE更新
        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        X_new = np.clip(X + (P + b - X) * dt + params['sigma'] * noise, 0, 1)

        repair_capacity = np.exp(-params['gamma'] * D)
        dD = (params['k_break'] * v_target * X_new * (1 - D) -
              (params['k_restore'] * triad_boost) * (1 - X_new) * D * repair_capacity) * dt
        D_new = np.clip(D + dD, 0, 1)

        X = X_new
        D = D_new

        mean_D_B = np.mean(D[N_A:])
        history_D_B.append(mean_D_B)

        # HJB 目的関数の積分: J = ∫ (α*D_B^2 + β*u^2) dt
        social_loss += params['alpha'] * (mean_D_B ** 2) * dt
        intervention_cost += params['beta'] * (u_t ** 2) * dt

    total_cost = social_loss + intervention_cost
    return total_cost, social_loss, intervention_cost, history_D_B[-1]

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000, 10000000]
    T_prevent_list = [10, 20, 30, 40, 50, 60, 70, 80] # 介入タイミングのスウィープ
    m = 2

    params = {
        'dt': 0.1,
        'steps': 150,
        'T_expose': 50,
        'sigma': 0.05,
        'k_break': 0.6,
        'k_restore': 0.1,
        'gamma': 6.0,
        'triad_multiplier': 5.0,
        'u_cost_weight': 1.0, # 単位時間あたりのアルゴリズム介入量
        'alpha': 100.0,       # 社会的分断に対するペナルティの重み
        'beta': 2.0           # アルゴリズム計算コストに対する重み
    }

    output_dir = "tmfg_hjb_results"
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for N in N_list:
        print(f"\n[{N} Nodes] ネットワーク生成中...")
        try:
            A_sparse, N_A, hub_A, boundary_nodes = build_clustered_network(N, m)
            print(f"-> 最適介入タイミング探索中 (HJB Cost Evaluation)...")

            costs = []
            for T_prev in tqdm(T_prevent_list, leave=False):
                t_cost, s_loss, i_cost, final_D = run_hjb_simulation(
                    N, A_sparse, N_A, hub_A, boundary_nodes, params, T_prev
                )
                costs.append({
                    'N': N,
                    'T_prevent': T_prev,
                    'Total_Cost': t_cost,
                    'Social_Loss': s_loss,
                    'Intervention_Cost': i_cost,
                    'Final_D_B': final_D
                })

            del A_sparse
            gc.collect()

            df_costs = pd.DataFrame(costs)
            csv_path = os.path.join(output_dir, f'hjb_cost_N{N}.csv')
            df_costs.to_csv(csv_path, index=False)
            all_results.extend(costs)

            # コストカーブの描画 (U字カーブの確認)
            plt.figure(figsize=(9, 6))
            plt.plot(df_costs['T_prevent'], df_costs['Total_Cost'], 'k-o', linewidth=2, label='Total Cost $J(u)$')
            plt.plot(df_costs['T_prevent'], df_costs['Social_Loss'], 'r--s', label='Social Loss (Metastasis)')
            plt.plot(df_costs['T_prevent'], df_costs['Intervention_Cost'], 'b-^', label='Intervention Cost')
            plt.axvline(x=params['T_expose'], color='gray', linestyle=':', label='T_expose (Phase Transition)')

            # 最適タイミング（コスト最小点）のマーキング
            opt_idx = df_costs['Total_Cost'].idxmin()
            opt_T = df_costs.loc[opt_idx, 'T_prevent']
            plt.axvline(x=opt_T, color='green', linestyle='-', alpha=0.5, label=f'Optimal $t^*={opt_T}$')

            plt.title(f'HJB Optimal Control Cost Curve (N={N:,})')
            plt.xlabel('Intervention Timing $T_{prevent}$')
            plt.ylabel('Cost Value')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'hjb_curve_N{N}.png'), dpi=300, bbox_inches='tight')
            plt.close()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # サマリーとZIP化
    df_all = pd.DataFrame(all_results)
    df_all.to_csv(os.path.join(output_dir, 'hjb_summary_all.csv'), index=False)

    zip_filename = 'TMFG_HJB_Optimization_Results.zip'
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                zipf.write(os.path.join(root, file), arcname=file)

    print(f"\n🎉 完了: 結果を '{zip_filename}' に出力しました。")

if __name__ == "__main__":
    main()


[1000 Nodes] ネットワーク生成中...
-> 最適介入タイミング探索中 (HJB Cost Evaluation)...



[10000 Nodes] ネットワーク生成中...
-> 最適介入タイミング探索中 (HJB Cost Evaluation)...



[100000 Nodes] ネットワーク生成中...
-> 最適介入タイミング探索中 (HJB Cost Evaluation)...



[1000000 Nodes] ネットワーク生成中...
-> 最適介入タイミング探索中 (HJB Cost Evaluation)...



[10000000 Nodes] ネットワーク生成中...
-> 最適介入タイミング探索中 (HJB Cost Evaluation)...



🎉 完了: 結果を 'TMFG_HJB_Optimization_Results.zip' に出力しました。
